In [1]:
import pandas as pd
import numpy as np
import os, sys
from tqdm.notebook import tqdm
from pathlib import Path
from plotly import express as px

In [2]:
class Estimator:
    def __init__(self):
        pass

    def fit(self):
        pass

    def __call__(self, well_data:pd.DataFrame, typewell_data:pd.DataFrame)->pd.Series:
        """Returns a TVT pd.Series"""
        pass

class Evaluator:
    def __init__(self):
        pass

    def evaluate_estimator(self, estimator:Estimator, file_path:Path=Path("rogii-wellbore-geology-prediction/train",)):
        well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in file_path.glob("*__horizontal_well.csv")
            }
        tvt_trues, tvt_preds= {}, {}
        for fname, paths in tqdm(well_files.items()):
            well_data, typewell_data = pd.read_csv(paths["Well"]), pd.read_csv(paths["TypeWell"])
            first_pred_idx = well_data.TVT_input.shift(1).dropna().index[-1]
            tvt_trues[fname] = well_data.loc[first_pred_idx:,"TVT"]
            tvt_preds[fname] = estimator(well_data)
        tvt_trues, tvt_preds = pd.DataFrame(tvt_trues), pd.DataFrame(tvt_preds)
        tvt_errors = tvt_preds - tvt_trues
        rmse = np.nanmean(tvt_errors.values.flatten()**2)**0.5
        return tvt_trues, tvt_preds, tvt_errors, rmse
    
    def make_submission(self, estimator:Estimator, file_path:Path=Path("rogii-wellbore-geology-prediction/test"),submission_file="submission.csv"):
        well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in file_path.glob("*__horizontal_well.csv")
            }
        tvt_preds = []
        for fname, paths in tqdm(well_files.items()):
            well_data, typewell_data = pd.read_csv(paths["Well"]), pd.read_csv(paths["TypeWell"])
            tvt_pred = estimator(well_data).rename("tvt")
            tvt_pred.index = pd.Series(fname + "_"+tvt_pred.index.values.astype(str),name="id")
            tvt_preds.append(tvt_pred)
        tvt_preds = pd.concat(tvt_preds,axis=0)
        tvt_preds.to_csv(submission_file)
        

In [3]:
class NaiveEstimator(Estimator):
    def __call__(self, well_data:pd.DataFrame, typewell_data:pd.DataFrame=None)->pd.Series:
        first_pred_idx = well_data.TVT_input.shift(1).dropna().index[-1]
        last_tvt = well_data.TVT_input.dropna().iloc[-1]
        pred_tvt = pd.Series(last_tvt,index=well_data.index[first_pred_idx:])
        return pred_tvt
        
        

In [4]:
evaluator = Evaluator()
estimator = NaiveEstimator()

In [5]:
filepath = Path("rogii-wellbore-geology-prediction/train",)
well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in filepath.glob("*__horizontal_well.csv")
            }

In [ ]:
naive_evaluation = evaluator.evaluate_estimator(estimator)
rmses = ((naive_evaluation[2]**2).mean(axis=0)**0.5).sort_values(ascending=False)
rmses.head(20)

  0%|          | 0/773 [00:00<?, ?it/s]

# Z Deviance
Assumes the wellbore does mostly a good job of following the formation.

In [ ]:
fname = "06df5958"
print(rmses.loc[fname])
well_data = pd.read_csv(well_files[fname]["Well"])
typewell_data = pd.read_csv(well_files[fname]["TypeWell"])
pred_start = well_data.TVT_input.shift(1).dropna().index[-1]

3.08848600855172


In [ ]:
# well_data = well_data.loc[pred_start:]
fig = px.scatter(well_data, y="Z",x="MD",color="TVT").update_layout(height=800,width=800,showlegend=False)
fig.update_traces(marker=dict(size=2))
start_point = well_data.loc[pred_start]
fig.add_trace(go.Scatter(
    x=[start_point["MD"]],
    y=[start_point["Z"]],
    mode="markers",
    marker=dict(color="red", size=8),
    name="Prediction Start",
    showlegend=False
))

fig.show()


In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
class ZDiffEstimator(Estimator):
    def __call__(self, well_data:pd.DataFrame, typewell_data:pd.DataFrame=None)->pd.Series:
        first_pred_idx = well_data.TVT_input.shift(1).dropna().index[-1]
        pred_data = well_data.loc[first_pred_idx:]
        X = pred_data.loc[:,["MD"]]
        z = pred_data.loc[:,"Z"]
        weights = np.ones(z.shape[0])
        weights[0] = 1e5
        zmodel = LinearRegression(fit_intercept=True).fit(X=X,y=z, sample_weight=weights)
        zpred = pd.Series(zmodel.predict(X), name="Zpred",index=z.index)
        zdiff = zpred - pred_data.Z
        last_tvt = well_data.TVT_input.dropna().iloc[-1]
        tvt_pred = zdiff + last_tvt
        return tvt_pred

        
estimator = ZDiffEstimator()

In [ ]:
tvt_trues, tvt_preds, tvt_errors, rmse = evaluator.evaluate_estimator(estimator)
rmses2 = ((tvt_errors**2).mean(axis=0)**0.5).sort_values(ascending=False)

  0%|          | 0/773 [00:00<?, ?it/s]